In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ============================================================
# 0. Device
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# 1. Load MNIST
# ============================================================
# MNIST images are 1x28x28. We'll interpret them as sequences
# of length 28, where each time step is a 28-dimensional vector.

batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),                 # [0, 255] -> [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # standard MNIST normalisation
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ============================================================
# 2. Define Simple RNN model for sequence classification
# ============================================================
# Input:  sequence length = 28, input_size = 28
# Model:  Embedding not needed (input is continuous pixels).
#         We'll feed each row as a 28-dim vector into RNN.
# Output: last hidden state -> linear -> 10 logits.

class RNNMnistClassifier(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=1, num_classes=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Simple RNN over the sequence of rows
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True  # input shape: (batch, seq_len, input_size)
        )

        # Final linear layer maps last hidden state -> class logits
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (B, 1, 28, 28)
        # Remove channel dimension and treat rows as time steps:
        # (B, 1, 28, 28) -> (B, 28, 28)
        x = x.squeeze(1)

        # RNN expects (B, seq_len, input_size) = (B, 28, 28)
        out, h_n = self.rnn(x)        # out: (B, 28, H), h_n: (num_layers, B, H)

        # Take last hidden state from the top layer
        last_hidden = h_n[-1]         # (B, H)

        # Map to class logits
        logits = self.fc(last_hidden) # (B, 10)
        return logits

model = RNNMnistClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# 3. Training and evaluation loops
# ============================================================
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

# ============================================================
# 4. Run training
# ============================================================
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)

    print(
        f"Epoch [{epoch}/{num_epochs}] "
        f"- Train loss: {train_loss:.4f}, acc: {train_acc:.4f} "
        f"- Test loss: {test_loss:.4f}, acc: {test_acc:.4f}"
    )

# ============================================================
# 5. Show a few predictions
# ============================================================
@torch.no_grad()
def show_example_predictions(model, loader, num_batches=1):
    model.eval()
    for i, (images, labels) in enumerate(loader):
        if i >= num_batches:
            break
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1)

        for j in range(10):  # show first 10 in this batch
            print(f"True: {labels[j].item()}, Pred: {preds[j].item()}")
        print("-----")
        break

print("\nExample predictions on test set:")
show_example_predictions(model, test_loader)


Using device: cpu


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.54MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.81MB/s]


Epoch [1/5] - Train loss: 0.8592, acc: 0.7134 - Test loss: 0.3762, acc: 0.8845
Epoch [2/5] - Train loss: 0.3260, acc: 0.9030 - Test loss: 0.2260, acc: 0.9357
Epoch [3/5] - Train loss: 0.2445, acc: 0.9289 - Test loss: 0.2037, acc: 0.9430
Epoch [4/5] - Train loss: 0.2043, acc: 0.9414 - Test loss: 0.1843, acc: 0.9461
Epoch [5/5] - Train loss: 0.1808, acc: 0.9477 - Test loss: 0.1681, acc: 0.9547

Example predictions on test set:
True: 7, Pred: 7
True: 2, Pred: 2
True: 1, Pred: 1
True: 0, Pred: 0
True: 4, Pred: 4
True: 1, Pred: 1
True: 4, Pred: 4
True: 9, Pred: 9
True: 5, Pred: 8
True: 9, Pred: 9
-----
